# Northstar Retail — Local Validation Demo

This notebook uses the packaged deterministic fixture and pandas Gold parity harness. It demonstrates locally verified structure and reconciliation without claiming that SQL Server, PostgreSQL, Databricks, or Power BI ran in this environment.

![Architecture](../docs/images/overall_architecture.svg)

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
SAMPLE = ROOT / "datasets" / "sample"
GOLD = ROOT / "datasets" / "demo_gold"
manifest = json.loads((GOLD / "gold_manifest.json").read_text())
manifest_summary = {k: manifest[k] for k in ["batch_id", "quality_status", "required_checks", "failed_required_checks"]}
manifest_summary

## Gold table inventory

In [ ]:
table_inventory = pd.DataFrame([
    {"table": row["table_name"], "rows": row["rows"], "sha256_prefix": row["sha256"][:12]}
    for row in manifest["tables"]
]).sort_values("table")
table_inventory

## Reconciled KPI sample

In [ ]:
sales = pd.read_csv(GOLD / "fact_sales.csv")
returns = pd.read_csv(GOLD / "fact_returns.csv")
shipments = pd.read_csv(GOLD / "fact_shipments.csv")
sessions = pd.read_csv(GOLD / "fact_web_sessions.csv")
spend = pd.read_csv(GOLD / "fact_marketing_spend.csv")

kpis = pd.Series({
    "Net Sales": sales.net_sales_amount.sum(),
    "Gross Profit": sales.gross_profit_amount.sum(),
    "Orders": sales.order_id.nunique(),
    "Valid Sales Lines": len(sales),
    "Returned Units": returns.return_quantity.sum(),
    "On-Time Delivery Rate": shipments.on_time_flag.mean(),
    "Web Sessions": sessions.session_count.sum(),
    "Conversion Rate": sessions.converted_flag.mean(),
    "Marketing Spend": spend.spend_amount.sum(),
})
kpis.to_frame("value")

## Source-to-Gold reconciliation

In [ ]:
reconciliation = pd.DataFrame(json.loads((GOLD / "reconciliation_results.json").read_text()))
reconciliation

## Monthly net-sales trend

In [ ]:
dates = pd.read_csv(GOLD / "dim_date.csv", parse_dates=["full_date"])
monthly = (sales.merge(dates[["date_key", "full_date"]], left_on="order_date_key", right_on="date_key")
           .assign(month=lambda d: d.full_date.dt.to_period("M").astype(str))
           .groupby("month", as_index=False).net_sales_amount.sum())
plt.figure(figsize=(10, 5))
plt.plot(monthly["month"], monthly["net_sales_amount"])
plt.xticks(rotation=90)
plt.xlabel("Month")
plt.ylabel("Net sales")
plt.title("Northstar Retail Monthly Net Sales — Deterministic Sample")
plt.tight_layout()
plt.show()

## Interpretation boundary

These values are generated from fictional data and are evidence that the local implementation is internally consistent. They are not real company performance and are not evidence that an external Databricks Job, SQL Server publication, or Power BI refresh succeeded.